In [ ]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage.filters import frangi
from skimage.morphology import skeletonize

# 1. Pipeline adapted for real medical imaging
class VesselSegmentationPipeline:
    def __init__(self, frangi_scales=(1.0, 8.0), frangi_step=0.5, 
                 morphology_kernel_size=3, vessel_threshold=0.01):
        # Adjusted scales for real vessels (often thinner than synthetic ones)
        self.frangi_scales = np.arange(frangi_scales[0], frangi_scales[1] + frangi_step, frangi_step)
        self.kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (morphology_kernel_size, morphology_kernel_size))
        self.vessel_threshold = vessel_threshold
    
    def segment(self, image):
        # Normalization is CRITICAL for Frangi on float64 data
        img_min, img_max = np.min(image), np.max(image)
        norm_image = (image - img_min) / (img_max - img_min + 1e-8)
        
        # Apply Frangi filter
        vesselness = frangi(norm_image, sigmas=self.frangi_scales, black_ridges=False)
        binary = (vesselness > self.vessel_threshold).astype(np.uint8) * 255
        
        # Morphological cleaning
        opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN, self.kernel)
        closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, self.kernel)
        
        # Keep only the largest connected component
        num_labels, labels = cv2.connectedComponents(closed)
        if num_labels > 1:
            vessel_mask = (labels == (1 + np.argmax(np.bincount(labels.ravel())[1:]))).astype(np.uint8) * 255
        else:
            vessel_mask = closed
            
        skeleton = skeletonize(vessel_mask > 0).astype(np.uint8) * 255
        return vessel_mask, skeleton, vesselness

# 2. Setup and Execution
data_folder = './logs/'  # Folder from your image
pipeline = VesselSegmentationPipeline(vessel_threshold=0.04)

# Filter for the .npz files listed in your directory
npz_files = [f for f in os.listdir(data_folder) if f.endswith('.npz')]

for npz_file in npz_files:
    # Load the .npz file
    data = np.load(os.path.join(data_folder, npz_file))
    volume = data['pixel_array'] # Shape is (Slices, 512, 512)
    
    # Process a Maximum Intensity Projection (MIP) to capture the whole vessel tree
    # This turns the 3D stack into a single 2D image for centerline extraction
    mip_image = np.max(volume, axis=0)
    
    # Run the pipeline
    mask, skeleton, vesselness = pipeline.segment(mip_image)

    # Visualization
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(mip_image, cmap='gray')
    axes[0].set_title(f'Original (MIP): {npz_file}')
    
    axes[1].imshow(vesselness, cmap='hot')
    axes[1].set_title('Vesselness Map')
    
    axes[2].imshow(mask, cmap='gray')
    axes[2].set_title('Segmented Mask')
    
    axes[3].imshow(skeleton, cmap='magma')
    axes[3].set_title('Extracted Centerline')
    
    plt.tight_layout()
    plt.show()

In [ ]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage.filters import frangi
from skimage.morphology import skeletonize

# 1. Updated Pipeline with Medical Preprocessing
class VesselSegmentationPipeline:
    def __init__(self, frangi_scales=(1.0, 5.0), frangi_step=0.5, 
                 morphology_kernel_size=5, vessel_threshold=0.04):
        # REDUCED SCALES TO TARGET THINNER VESSELS
        self.frangi_scales = np.arange(frangi_scales[0], frangi_scales[1] + frangi_step, frangi_step)
        # LARGER KERNEL FOR BETTER NOISE CLEANING
        self.kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (morphology_kernel_size, morphology_kernel_size))
        self.vessel_threshold = vessel_threshold
    
    def segment(self, image):
        # --- NEW: CROP EDGES TO REMOVE DETECTOR BORDER ARTIFACTS ---
        h, w = image.shape
        margin = 40 
        cropped = image[margin:h-margin, margin:w-margin]
        
        # --- NEW: ENHANCE CONTRAST USING CLAHE ---
        # CONVERT TO UINT8 FOR OPENCV COMPATIBILITY
        img_uint8 = cv2.normalize(cropped, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
        enhanced = clahe.apply(img_uint8)
        
        # --- NEW: NORMALIZE ENHANCED IMAGE FOR FRANGI ---
        norm_image = enhanced / 255.0
        
        # --- UPDATED: FRANGI FILTER SET TO DETECT DARK VESSELS (BLACK_RIDGES=True) ---
        vesselness = frangi(norm_image, sigmas=self.frangi_scales, black_ridges=True)
        
        # THRESHOLDING
        binary = (vesselness > self.vessel_threshold).astype(np.uint8) * 255
        
        # MORPHOLOGICAL CLEANING
        cleaned = cv2.morphologyEx(binary, cv2.MORPH_OPEN, self.kernel)
        
        # SKELETONIZATION
        skeleton = skeletonize(cleaned > 0).astype(np.uint8) * 255
        
        # --- NEW: RECONSTRUCT FULL IMAGE SIZE BY ADDING BLACK PADDING BACK ---
        full_skeleton = np.zeros_like(image)
        full_skeleton[margin:h-margin, margin:w-margin] = skeleton
        
        return cleaned, full_skeleton, vesselness

# 2. Execution Logic
# INCREASE THRESHOLD IF YOU STILL GET MESSY LINES
pipeline = VesselSegmentationPipeline(vessel_threshold=0.05)

data_dir = './logs/LCA' 
npz_files = [f for f in os.listdir(data_dir) if f.endswith('.npz')]

if not npz_files:
    print(f"Check your path! No .npz files found in: {os.path.abspath(data_dir)}")

for file_name in npz_files:
    file_path = os.path.join(data_dir, file_name)
    data = np.load(file_path)
    
    # LOAD PIXEL ARRAY (SHAPES LIKE 45, 512, 512) [cite: 1, 9]
    volume = data['pixel_array'] 
    
    # MAXIMUM INTENSITY PROJECTION (MIP) TO FLATTEN 3D TO 2D
    mip_image = np.max(volume, axis=0)
    
    # RUN UPDATED PIPELINE
    mask, skeleton, vesselness = pipeline.segment(mip_image)

    # VISUALIZATION
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(mip_image, cmap='gray')
    axes[0].set_title(f"Original (MIP): {file_name}")
    
    axes[1].imshow(vesselness, cmap='hot')
    axes[1].set_title("Enhanced Frangi Vesselness")
    
    axes[2].imshow(skeleton, cmap='magma')
    axes[2].set_title("Cleaned Centerline")
    
    plt.tight_layout()
    plt.show()

In [ ]:
import os
import glob
import cv2
import numpy as np
import imageio
from base64 import b64encode
from skimage.filters import frangi
from skimage.morphology import skeletonize, remove_small_objects
from IPython.display import Image, display, HTML

class MultiFramePipeline:
    def __init__(self, vessel_threshold=0.04, min_size=200):
        self.vessel_threshold = vessel_threshold
        self.min_size = min_size

    def segment_single_frame(self, frame):
        h, w = frame.shape
        m = 45 
        img = frame[m:h-m, m:w-m]
        img_8 = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        denoised = cv2.medianBlur(img_8, 3)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        enhanced = clahe.apply(denoised)
        v = frangi(enhanced/255.0, sigmas=np.arange(2, 5, 1), black_ridges=True)
        mask = v > self.vessel_threshold
        clean_mask = remove_small_objects(mask, min_size=self.min_size)
        skel = skeletonize(clean_mask).astype(np.uint8) * 255
        full_skel = np.zeros_like(frame, dtype=np.uint8)
        full_skel[m:h-m, m:w-m] = skel
        return full_skel

# --- CONFIGURATION ---
input_folder = './logs/LCA/'
output_folder = './logs/processed_outputs/'
os.makedirs(output_folder, exist_ok=True)

# Find all .npz files in the directory
all_files = glob.glob(os.path.join(input_folder, "*.npz"))
pipeline = MultiFramePipeline(vessel_threshold=0.04, min_size=200)

print(f"FOUND {len(all_files)} FILES TO PROCESS.\n")

for data_path in all_files:
    # Get the base name of the file (e.g., '9_1.3.46...')
    base_name = os.path.basename(data_path).replace('.npz', '')
    print(f"--- STARTING: {base_name} ---")
    
    # Load data
    data = np.load(data_path)
    volume = data['pixel_array']
    
    all_frames = []
    for i in range(len(volume)):
        frame_skel = pipeline.segment_single_frame(volume[i])
        orig_norm = cv2.normalize(volume[i], None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        visible_skel = cv2.dilate(frame_skel, np.ones((2,2), np.uint8), iterations=1)
        combined = np.hstack((orig_norm, visible_skel))
        all_frames.append(combined)

    # Define unique output names
    out_gif = os.path.join(output_folder, f"{base_name}.gif")
    out_mp4 = os.path.join(output_folder, f"{base_name}.mp4")

    # 1. Save and Display GIF
    imageio.mimsave(out_gif, all_frames, fps=15, loop=0)
    display(HTML(f"<b>GIF Saved:</b> {out_gif}"))
    display(Image(filename=out_gif, width=500))

    # 2. Save and Display MP4 (Fixed with FFMPEG format)
    imageio.mimsave(out_mp4, all_frames, format='FFMPEG', fps=15, quality=8, macro_block_size=1)
    
    mp4_data = open(out_mp4, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4_data).decode()
    
    display(HTML(f"""
        <b>Video Player:</b> {out_mp4}<br>
        <video width="500" controls loop autoplay>
              <source src="{data_url}" type="video/mp4">
        </video>
        <hr style="border: 2px solid #000;">
    """))

print("ALL FILES PROCESSED SUCCESSFULLY!")